[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_6_8/12_tag_6_8_xai_overfitting_spurious_logo.ipynb)

# Tag 6-8 - 12: XAI an einem spurious Feature in CIFAR-10

Dieses Beispiel nutzt **echte CIFAR-10-Bilder**: Katze gegen Hund. Es zeigt keine globale, gemittelte Erklärung. Stattdessen betrachten wir mehrere **konkrete Einzelentscheidungen** und testen dieselben Testbilder unter drei Bedingungen.

Wir erzeugen absichtlich einen Datenfehler: Im Training erhält jedes Hundebild unten rechts ein kleines magentafarbenes Logo; Katzenbilder nicht. Das Logo ist daher ein perfekter, aber sachlich falscher Hinweis. Ein Modell kann sich diese Abkürzung merken, statt Katze und Hund zu unterscheiden.

> Voraussetzung: Zuerst Notebook `00_tag_6_8_optionale_datensaetze_vorbereiten.ipynb` ausführen. Dieses Notebook lädt nichts aus dem Internet.

## Versuchsaufbau

- **Training/Validierung:** 450/150 CIFAR-10-Bilder je Klasse; bei *dog* ist das Logo immer vorhanden.
- **Test:** die offiziellen, davon getrennten CIFAR-10-Testbilder. Die zugrunde liegenden Bilder bleiben in allen drei Varianten identisch. Nur das Logo ändert sich.
- **Gleicher Bias:** Logo bei Hund wie im Training.
- **Ohne Logo:** kein Bild hat ein Logo.
- **Bias umgedreht:** Logo bei Katze statt bei Hund.

Eine hohe Accuracy im Test *gleicher Bias* ist daher keine ausreichende Evidenz für eine sinnvolle Entscheidung. Besonders aussagekräftig sind der Einbruch ohne Logo bzw. beim umgedrehten Bias und die lokalen Saliency-Maps weiter unten.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("datasets")
BATCH_SIZE = 64
EPOCHS = 8
CLASS_IDS = (3, 5)  # CIFAR-10: cat, dog
CLASS_NAMES = ("cat", "dog")

plt.rcParams["axes.grid"] = False
print("PyTorch:", torch.__version__)
print("Device:", device)
print("Datenordner:", DATA_DIR.resolve())

## Daten laden und sauber aufteilen

Die Validierung kommt aus dem offiziellen Trainingssplit. Für die abschließende Diagnose verwenden wir ausschließlich den offiziellen Testsplit. So werden weder Testbilder noch Testbedingungen zur Modellauswahl verwendet.

In [ ]:
to_tensor = transforms.ToTensor()
try:
    cifar_train = datasets.CIFAR10(DATA_DIR, train=True, download=False, transform=to_tensor)
    cifar_test = datasets.CIFAR10(DATA_DIR, train=False, download=False, transform=to_tensor)
except RuntimeError as error:
    raise RuntimeError(
        "CIFAR-10 fehlt im lokalen Cache. Bitte zuerst Notebook 00 ausführen."
    ) from error


def select_per_class(dataset, class_ids, n_per_class, seed):
    """Wählt pro Klasse zufällig feste Indizes; die Zufallswahl ist reproduzierbar."""
    rng = np.random.default_rng(seed)
    targets = np.asarray(dataset.targets)
    selected = []
    for class_id in class_ids:
        candidates = np.flatnonzero(targets == class_id)
        if len(candidates) < n_per_class:
            raise ValueError(f"Zu wenige Bilder für Klasse {class_id}.")
        selected.extend(rng.choice(candidates, size=n_per_class, replace=False).tolist())
    rng.shuffle(selected)
    return selected


class BinaryCIFAR(Dataset):
    """Zwei CIFAR-10-Klassen mit lokalen Labels 0=cat und 1=dog."""
    def __init__(self, base_dataset, indices, class_ids=CLASS_IDS):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.label_map = {class_id: local_label for local_label, class_id in enumerate(class_ids)}
        self.class_names = CLASS_NAMES

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        image, original_label = self.base_dataset[self.indices[index]]
        return image, torch.tensor(self.label_map[int(original_label)], dtype=torch.long)


# 600 Bilder pro Klasse aus dem Trainingssplit: 450 Training, 150 Validierung.
train_valid_indices = select_per_class(cifar_train, CLASS_IDS, n_per_class=600, seed=RANDOM_STATE)
train_indices = []
valid_indices = []
for class_id in CLASS_IDS:
    own = [i for i in train_valid_indices if cifar_train.targets[i] == class_id]
    train_indices.extend(own[:450])
    valid_indices.extend(own[450:])

# Der Test bleibt komplett vom offiziellen Testsplit getrennt.
test_indices = select_per_class(cifar_test, CLASS_IDS, n_per_class=1_000, seed=RANDOM_STATE)
clean_train = BinaryCIFAR(cifar_train, train_indices)
clean_valid = BinaryCIFAR(cifar_train, valid_indices)
clean_test = BinaryCIFAR(cifar_test, test_indices)
print(f"Train: {len(clean_train)}, Validierung: {len(clean_valid)}, Test: {len(clean_test)}")

## Das kontrollierte Störmerkmal: ein Logo

Das Logo wird **erst beim Auslesen** auf den Tensor gezeichnet. Dadurch verwenden alle Varianten exakt dieselben Originalbilder; der Vergleich misst wirklich nur den Effekt des Logos.

In [ ]:
def add_logo(image):
    """Fügt unten rechts ein gut sichtbares 8x8-Magenta-Logo ein."""
    result = image.clone()
    result[0, -10:-2, -10:-2] = 1.0  # Rot
    result[1, -10:-2, -10:-2] = 0.0  # Grün
    result[2, -10:-2, -10:-2] = 1.0  # Blau
    result[:, -8:-4, -8:-4] = 0.0    # kleines dunkles Zentrum
    return result


class LogoDataset(Dataset):
    """Zeigt dieselben Bilder mit einer definierbaren Logo-Regel."""
    def __init__(self, base_dataset, logo_rule):
        self.base_dataset = base_dataset
        self.logo_rule = logo_rule
        self.class_names = base_dataset.class_names

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        image, label = self.base_dataset[index]
        has_logo = {
            "same_bias": int(label) == 1,
            "no_logo": False,
            "reversed_bias": int(label) == 0,
        }[self.logo_rule]
        return (add_logo(image) if has_logo else image), label


train_ds = LogoDataset(clean_train, "same_bias")
valid_ds = LogoDataset(clean_valid, "same_bias")
test_same_bias = LogoDataset(clean_test, "same_bias")
test_no_logo = LogoDataset(clean_test, "no_logo")
test_reversed_bias = LogoDataset(clean_test, "reversed_bias")

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for col in range(4):
    clean_image, label = clean_train[col]
    biased_image, _ = train_ds[col]
    axes[0, col].imshow(clean_image.permute(1, 2, 0))
    axes[1, col].imshow(biased_image.permute(1, 2, 0))
    axes[0, col].set_title(CLASS_NAMES[int(label)])
    axes[0, col].axis("off")
    axes[1, col].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Training")
fig.suptitle("Im Training erhalten nur Hundebilder das Logo", y=1.02)
plt.tight_layout()

## Kleines CNN trainieren

Wir trainieren absichtlich ohne Augmentation oder Regularisierung. Das ist hier kein empfehlenswertes Produktionsmodell, sondern eine kontrollierte Situation, in der eine bequeme Abkürzung besonders attraktiv ist. Die Lernkurve nutzt nur Training und die gleichartig markierte Validierung.

In [ ]:
def make_loader(dataset, shuffle=False):
    return DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )


train_loader = make_loader(train_ds, shuffle=True)
valid_loader = make_loader(valid_ds)


class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(64, 2)

    def forward(self, image):
        return self.classifier(self.features(image).flatten(1))


def evaluate(model, loader, loss_fn=None):
    model.eval()
    correct = total = 0
    losses = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            if loss_fn is not None:
                losses.append(loss_fn(logits, labels).item() * len(labels))
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += len(labels)
    return (sum(losses) / total if losses else np.nan), correct / total


model = SmallCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = train_correct = train_total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * len(labels)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_total += len(labels)
    valid_loss, valid_acc = evaluate(model, valid_loader, loss_fn)
    history.append({
        "Epoche": epoch,
        "Train-Accuracy": train_correct / train_total,
        "Validierungs-Accuracy": valid_acc,
        "Train-Loss": train_loss_sum / train_total,
        "Validierungs-Loss": valid_loss,
    })

history_df = pd.DataFrame(history)
ax = history_df.plot(x="Epoche", y=["Train-Accuracy", "Validierungs-Accuracy"], marker="o", ylim=(0, 1.05))
ax.set_ylabel("Accuracy")
ax.set_title("Lernkurve unter derselben (fehlerhaften) Logo-Regel")
plt.show()

## Derselbe Test, drei Logo-Bedingungen

Es gibt hier nicht drei unterschiedliche Testsätze: Jedes der 2.000 Original-Testbilder erscheint in jeder Zeile. Wenn die Accuracy beim Entfernen oder Umkehren des Logos deutlich sinkt, hat das Modell die Korrelation aus dem Training ausgenutzt.

In [ ]:
test_conditions = {
    "Logo wie im Training": test_same_bias,
    "Logo entfernt": test_no_logo,
    "Logo bei Katze (umgedreht)": test_reversed_bias,
}

results = []
for condition, dataset in test_conditions.items():
    _, accuracy = evaluate(model, make_loader(dataset), loss_fn)
    results.append({"Testbedingung": condition, "Accuracy": accuracy})
results_df = pd.DataFrame(results)
display(results_df.style.format({"Accuracy": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(results_df["Testbedingung"], results_df["Accuracy"], color=["#4C78A8", "#F58518", "#E45756"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy")
ax.set_title("Generalisierung, wenn sich nur das Logo ändert")
ax.tick_params(axis="x", rotation=10)
for index, accuracy in enumerate(results_df["Accuracy"]):
    ax.text(index, accuracy + 0.03, f"{accuracy:.1%}", ha="center")
plt.tight_layout()

## Lokale XAI: einzelne Entscheidungen prüfen

Saliency beantwortet hier die konkrete Frage: **Welche Pixel beeinflussen den Score der gerade vorhergesagten Klasse in diesem Bild?** Warme Bereiche im Overlay zeigen Pixel, auf die der Score lokal besonders empfindlich reagiert.

Wir zeigen bewusst mehrere Beispiele statt einer globalen Mittelung: oben drei korrekte Entscheidungen mit dem vertrauten Logo, darunter möglichst drei falsche Entscheidungen beim umgedrehten Bias. Falls das Modell dort überraschend keine falschen Entscheidungen macht, werden stattdessen beliebige Beispiele dieser Bedingung gezeigt.

In [ ]:
def prediction(model, image):
    model.eval()
    with torch.no_grad():
        probabilities = model(image.unsqueeze(0).to(device)).softmax(dim=1)[0].cpu()
    predicted = int(probabilities.argmax())
    return predicted, float(probabilities[predicted])


def saliency_map(model, image, target_class):
    """Gradient des vorhergesagten Klassen-Scores nach den Eingabepixeln."""
    model.eval()
    input_image = image.unsqueeze(0).to(device).clone().detach().requires_grad_(True)
    model.zero_grad(set_to_none=True)
    score = model(input_image)[0, target_class]
    score.backward()
    saliency = input_image.grad.abs().max(dim=1).values[0].detach().cpu().numpy()
    return saliency / max(saliency.max(), 1e-8)


def find_examples(dataset, want_correct, count=3):
    found = []
    for index in range(len(dataset)):
        image, label = dataset[index]
        predicted, _ = prediction(model, image)
        if (predicted == int(label)) == want_correct:
            found.append(index)
        if len(found) == count:
            return found
    return found


def plot_local_explanations(dataset, indices, title):
    fig, axes = plt.subplots(len(indices), 3, figsize=(8, 2.5 * len(indices)), squeeze=False)
    for row, index in enumerate(indices):
        image, label = dataset[index]
        predicted, confidence = prediction(model, image)
        heatmap = saliency_map(model, image, predicted)
        image_hwc = image.permute(1, 2, 0).numpy()
        text = f"wahr: {CLASS_NAMES[int(label)]}\nvorhergesagt: {CLASS_NAMES[predicted]} ({confidence:.0%})"

        axes[row, 0].imshow(image_hwc)
        axes[row, 0].set_title(text, fontsize=9)
        axes[row, 1].imshow(heatmap, cmap="magma", vmin=0, vmax=1)
        axes[row, 1].set_title("lokale Saliency", fontsize=9)
        axes[row, 2].imshow(image_hwc)
        axes[row, 2].imshow(heatmap, cmap="magma", alpha=0.55, vmin=0, vmax=1)
        axes[row, 2].set_title("Overlay", fontsize=9)
        for axis in axes[row]:
            axis.axis("off")
    fig.suptitle(title, y=1.01)
    plt.tight_layout()


same_examples = find_examples(test_same_bias, want_correct=True)
reversed_errors = find_examples(test_reversed_bias, want_correct=False)
if len(reversed_errors) < 3:
    reversed_errors = list(range(3))

plot_local_explanations(test_same_bias, same_examples, "Drei lokale Erklärungen: Test mit bekanntem Logo")
plot_local_explanations(test_reversed_bias, reversed_errors, "Drei lokale Erklärungen: Logo-Regel umgedreht")

## Eine Gegenfaktische Probe: exakt dasselbe Bild mit und ohne Logo

Zum Schluss nehmen wir ein einzelnes Katzen-Testbild und verändern **nur** die Ecke. Ändert sich die Vorhersage deutlich, ist das besonders anschauliche Evidenz für das spurious Feature. Das ist keine globale Aussage über das Modell, sondern eine nachvollziehbare lokale Prüfung.

In [ ]:
# Erstes Katzenbild im offiziellen Testsplit suchen.
cat_index = next(i for i in range(len(clean_test)) if int(clean_test[i][1]) == 0)
clean_image, label = clean_test[cat_index]
logo_image = add_logo(clean_image)

fig, axes = plt.subplots(2, 3, figsize=(8, 5))
for row, (name, image) in enumerate([("ohne Logo", clean_image), ("mit Logo", logo_image)]):
    predicted, confidence = prediction(model, image)
    heatmap = saliency_map(model, image, predicted)
    image_hwc = image.permute(1, 2, 0).numpy()
    axes[row, 0].imshow(image_hwc)
    axes[row, 0].set_title(f"{name}\nVorhersage: {CLASS_NAMES[predicted]} ({confidence:.0%})")
    axes[row, 1].imshow(heatmap, cmap="magma", vmin=0, vmax=1)
    axes[row, 1].set_title("lokale Saliency")
    axes[row, 2].imshow(image_hwc)
    axes[row, 2].imshow(heatmap, cmap="magma", alpha=0.55, vmin=0, vmax=1)
    axes[row, 2].set_title("Overlay")
    for axis in axes[row]:
        axis.axis("off")
fig.suptitle("Gegenfaktisch: Gleiches Katzenbild, nur das Logo unterscheidet sich", y=1.02)
plt.tight_layout()

## Einordnung

- Ein Einbruch in den beiden veränderten Testbedingungen zeigt: Gute Accuracy unter dem bekannten Bias war nicht gleichbedeutend mit robuster Objekterkennung.
- Saliency ist ein Hinweis auf wichtige Pixel, kein Beweis für Kausalität. Die kontrollierten Gegenfaktischen (gleiches Bild, nur Logo geändert) ergänzen sie deshalb sinnvoll.
- Mögliche Gegenmaßnahmen: Logo/Wasserzeichen entfernen oder über Klassen gleich verteilen, Daten aus mehreren Quellen verwenden, geeignete Augmentation wie Random Erasing einsetzen und solche Bias-Tests als festen Teil der Evaluation aufnehmen.